In [1]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from dotenv import load_dotenv
load_dotenv(Path("..") / ".env", override=True)

import mlflow
import numpy as np
import pandas as pd
from mlflow_settings import configure_mlflow

configure_mlflow()

client = mlflow.MlflowClient()

models = {}
models["per_sqm"] = mlflow.pyfunc.load_model("models:/RealEstatePricePerSqm/latest")

sample = pd.DataFrame([{
    "size_m2": 75.0,
    "nr_of_rooms": 3,
    "floor": 7,
    "building_total_floors": 8,
    "neighbourhood": "Редута",
    "is_first_floor": 0,
    "is_last_floor": 0,
    "is_furnished": 0,
    "near_public_transport": 0,
}])

log_pred  = models["per_sqm"].predict(sample)[0]
price_eur = np.exp(log_pred)
print(f"\nPredicted log-price:    {log_pred:.4f}")
print(f"Predicted price (EUR):  {price_eur:,.0f}")



Predicted log-price:    8.1445
Predicted price (EUR):  3,445


### Test the 'Happy Path'

In [7]:
from fastapi.testclient import TestClient
from api.inference_service import app

def test_predict_total_price_returns_valid_output():
    '''Test that the model returns output correctly'''

    payload = {
    "size_m2": 75.0,
    "nr_of_rooms": 3,
    "floor": 7,
    "building_total_floors": 8,
    "neighbourhood": "Редута",
    "is_first_floor": 0,
    "is_last_floor": 0,
    "is_furnished": 0,
    "near_public_transport": 0
    }
    with TestClient(app) as client:
        r = client.post("/predictPricePerSqm", json=payload)

    # return assertion error if http response code != 200 or results is negative value 
    assert r.status_code == 200
    assert r.json()["normalized_result"] > 0

test_predict_total_price_returns_valid_output()

### Test input validation
We will test if the model responds correctly to the following:
- missing input feature
- wrong dtype

In [ ]:
from unittest.mock import patch, MagicMock

def test_predict_price_mocked_model():
    mock_prediction = 450_000.0

    with patch("app.services.model.predict", return_value=mock_prediction):
        response = client.post("/predict/price", json={
        "size_m2": 75.0,
        "nr_of_rooms": 3,
        "floor": 7,
        "building_total_floors": 8,
        "neighbourhood": "Редута",
        "is_first_floor": 0,
        "is_last_floor": 0,
        "is_furnished": 0,
        "near_public_transport": 0
        })

    assert response.status_code == 200
    assert response.json()["predicted_price"] == mock_prediction

In [ ]:
from unittest.mock import MagicMock, patch
# from fastapi.testclient import TestClient
from api.inference_service import app
import numpy as np

SAMPLE_PAYLOAD = {
    "size_m2": 75.0,
    "nr_of_rooms": 3,
    "floor": 7,
    "building_total_floors": 8,
    "neighbourhood": "Lulin",
    "is_first_floor": 0,
    "is_last_floor": 0,
    "is_furnished": 0,
    "near_public_transport": 0,
}

def make_mock_model(log_value: float) -> MagicMock:
    """Return a mock that mimics mlflow.pyfunc.PyFuncModel.predict()"""
    mock = MagicMock()
    mock.predict.return_value = [log_value]
    return mock

def test_valid_payload_returns_positive_price():
    fake_models = {
        "per_sqm": make_mock_model(log_value=7.5),
        "total_price": make_mock_model(log_value=12.5),
    }
    with patch.dict("api.inference_service.models", fake_models):
        r = client.post("/predictPricePerSqm", json=SAMPLE_PAYLOAD)
    assert r.status_code == 200
    data = r.json()
    assert data["log_result"] == 7.5
    assert abs(data["normalized_result"] - np.exp(7.5)) < 1e-6
    print("PASSED — normalized_result:", data["normalized_result"])

test_valid_payload_returns_positive_price()

PASSED — normalized_result: 1808.0424144560632


In [ ]:
# INPUT VALIDATION — bad data, you expect rejection before the model even runs
WRONG_PAYLOAD = {
# "size_m2": 75.0,
"nr_of_rooms": 3,
"floor": 7,
"building_total_floors": 8,
"neighbourhood": "Lulin",
"is_first_floor": 0,
"is_last_floor": 0,
"is_furnished": 0,
"near_public_transport": 0,
}

def test_predict_price_missing_field():
    '''Test with mock model with missing input feature'''
    fake_models = {
        "per_sqm": make_mock_model(log_value=7.5),
        "total_price": make_mock_model(log_value=12.5),
    }

    with patch.dict("api.inference_service.models", fake_models):
        r = client.post("/predictPricePerSqm", json=WRONG_PAYLOAD)

    assert r.status_code == 422, f"Expected 422, got {r.status_code}: {r.json()}"
    print("PASSED missing field test!")

test_predict_price_missing_field()

PASSED missing field test!


In [ ]:
import pytest
'''This needs to be in .py file. Notebooks dont have pytest runner'''
# SAMPLE_PAYLOAD = {"f1": 1, "f2": "abc", "f3": 10.5}
all_fields = SAMPLE_PAYLOAD.keys()

@pytest.mark.parametrize("missing_field", all_fields)
def test_missing_field_returns_422(missing_field):
    fake_models = {
        "per_sqm": make_mock_model(log_value=7.5),
        "total_price": make_mock_model(log_value=12.5),
    }
    payload = {k: v for k, v in SAMPLE_PAYLOAD.items() if k != missing_field}

    with patch.dict("api.inference_service.models", fake_models):
        r = client.post("/predictPricePerSqm", json=payload)    
    assert r.status_code == 422

test_missing_field_returns_422()

TypeError: test_missing_field_returns_422() missing 1 required positional argument: 'missing_field'

In [24]:
for missing_field in SAMPLE_PAYLOAD.keys():
    fake_models = {
        "per_sqm": make_mock_model(log_value=7.5),
        "total_price": make_mock_model(log_value=12.5),
    }
    payload = {k: v for k, v in SAMPLE_PAYLOAD.items() if k != missing_field}

    with patch.dict("api.inference_service.models", fake_models):
        r = client.post("/predictPricePerSqm", json=payload)

    assert r.status_code == 422, f"Expected 422 when '{missing_field}' is missing, got {r.status_code}"
    print(f"✓ missing '{missing_field}' → 422")

✓ missing 'size_m2' → 422
✓ missing 'nr_of_rooms' → 422
✓ missing 'floor' → 422
✓ missing 'building_total_floors' → 422
✓ missing 'neighbourhood' → 422
✓ missing 'is_first_floor' → 422
✓ missing 'is_last_floor' → 422
✓ missing 'is_furnished' → 422
✓ missing 'near_public_transport' → 422
